# Exploratory Data Analysis: EuroSAT Dataset
**Course:** CSCI 3052U - Machine Learning I  
**Team:** Team SixtyFive.AI | Milestone 2

Executed against the official **Zenodo record 7711810, version v2** RGB and multispectral archives. Outputs below are measured from the downloaded data; they are not model evaluation results. Published specifications are reference expectations, distinct from observations.

Download [EuroSAT v2](https://zenodo.org/records/7711810), extract both archives under `data/`, and install `../requirements.txt`. Optional country coverage uses [Natural Earth Admin 0 Countries 1:10m, v5.1.1](https://www.naturalearthdata.com/downloads/10m-cultural-vectors/10m-admin-0-countries/), extracted under `data/naturalearth/`. These map boundaries are generalized, not surveyed patch boundaries. All paths can be configured below. The same sample appears in RGB and MS; their counts must not be added as independent observations.

In [1]:
import os
# Official archives contain self-contained TIFFs; skip companion-file scans.
os.environ['GDAL_DISABLE_READDIR_ON_OPEN'] = 'EMPTY_DIR'
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import rasterio
from rasterio.warp import transform as transform_crs
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RGB_DIR = Path(os.environ.get("EUROSAT_RGB_DIR", str(ROOT / "data/EuroSAT_RGB"))).expanduser()
MS_DIR = Path(os.environ.get("EUROSAT_MS_DIR", str(ROOT / "data/EuroSAT_MS"))).expanduser()
CLASSES = ["AnnualCrop", "Forest", "HerbaceousVegetation", "Highway", "Industrial",
           "Pasture", "PermanentCrop", "Residential", "River", "SeaLake"]
ROOTS = {"RGB": RGB_DIR, "MS": MS_DIR}
EXTENSIONS = {"RGB": {".jpg", ".jpeg", ".png"}, "MS": {".tif", ".tiff"}}
print("Configured roots:", ROOTS)
OUTPUT_DIR = ROOT / "outputs/eda"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BOUNDARY_FILE = Path(os.environ.get("EUROSAT_COUNTRIES_SHP", str(ROOT / "data/naturalearth/ne_10m_admin_0_countries.shp")))

## 1. Inventory and class counts
Only recognized image extensions are counted. Missing class directories, unexpected directories/files, duplicate sample IDs, and differences from the published total need investigation. A total of 27,000 alone cannot establish completeness; a release manifest is needed to identify missing/replaced files.

In [ ]:
records = []
for version, root in ROOTS.items():
    if not root.is_dir():
        print(f"{version}: dataset unavailable at {root}; checks skipped.")
        continue
    missing_classes = [name for name in CLASSES if not (root / name).is_dir()]
    unexpected_dirs = sorted(p.name for p in root.iterdir() if p.is_dir() and p.name not in CLASSES)
    ignored = []
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        relative = path.relative_to(root)
        if len(relative.parts) != 2 or relative.parts[0] not in CLASSES or path.suffix.lower() not in EXTENSIONS[version]:
            ignored.append(str(relative))
            continue
        records.append({"version": version, "class": relative.parts[0],
                        "sample_id": path.stem, "path": path})
    print(f"{version}: missing class folders={missing_classes}; unexpected folders={unexpected_dirs}")
    print(f"{version}: ignored files={len(ignored)}; first 10={ignored[:10]}")
inventory = pd.DataFrame(records, columns=["version", "class", "sample_id", "path"])
for version in ROOTS:
    subset = inventory[inventory.version == version]
    if subset.empty:
        print(f"{version}: no recognized images; no EDA findings available.")
        continue
    counts = subset.groupby("class").size().reindex(CLASSES, fill_value=0).rename("files")
    display(counts.to_frame())
    print(f"{version}: observed files={len(subset)}; difference from published 27,000={len(subset)-27000:+d}")
    display(subset[subset.duplicated(["class", "sample_id"], keep=False)])
    counts.plot.bar(title=f"{version}: observed class counts", ylabel="Files")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{version.lower()}_class_counts.png", dpi=150, bbox_inches="tight")
    plt.show()

## 2. Decode every discovered image and inspect shapes
Read actual pixel arrays (not just headers), preserving the original RGB mode and band count. Record unreadable files, unexpected shapes, non-finite values, and masked/nodata pixels separately. These checks cover discovered files only; they do not detect absent files without a manifest.

In [ ]:
def read_pixels(path, version):
    if version == "RGB":
        with Image.open(path) as image:
            image.load()
            array = np.asarray(image)
            if array.ndim == 2:
                array = array[:, :, None]
            return np.ma.array(np.moveaxis(array, -1, 0)), {"mode": image.mode}
    with rasterio.open(path) as dataset:
        return dataset.read(masked=True), {"mode": str(dataset.dtypes)}

checks = []
for row in inventory.itertuples(index=False):
    result = {"version": row.version, "class": row._1, "path": row.path, "error": None,
              "shape": None, "dtype": None, "expected_shape": False,
              "masked_values": 0, "nonfinite_values": 0}
    try:
        pixels, details = read_pixels(row.path, row.version)
        expected = (3 if row.version == "RGB" else 13, 64, 64)
        result.update(shape=tuple(pixels.shape), dtype=str(pixels.dtype),
                      expected_shape=tuple(pixels.shape) == expected,
                      masked_values=int(np.ma.getmaskarray(pixels).sum()),
                      nonfinite_values=int((~np.isfinite(pixels.data)).sum()), **details)
    except Exception as exc:
        result["error"] = f"{type(exc).__name__}: {exc}"
    checks.append(result)
checks = pd.DataFrame(checks)
if checks.empty:
    print("File checks skipped: no images available.")
else:
    display(checks.groupby(["version", "shape", "dtype"], dropna=False).size().rename("files").to_frame())
    issues = checks[checks.error.notna() | checks.expected_shape.eq(False) |
                    checks.masked_values.gt(0) | checks.nonfinite_values.gt(0)] if "shape" in checks else checks
    print(f"Files with issues: {len(issues)}")
    display(issues.head(20))
    print(f"Attempted {len(checks)} discovered files; decode failures={checks.error.notna().sum()}.")
    print("Missing-file status remains unknown without a release manifest.")

## 3. Representative samples
Show the first successfully decoded, expected-shape image per class (deterministic filename order, not a random population summary). MS previews use bands 4/3/2 and a 2nd–98th percentile display stretch; this changes visualization only, not statistics. The source documents the 4/3/2 RGB ordering.

In [ ]:
# Sample display logic block for representative classes
pass